<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/MC13_SOC_Assistant_LangChain_Simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MC 13: Building AI Agents with LangChain
### Build a SOC Assistant for an energy plant

**Instructor:** Mohammad Arshad Ahmed · Decoding Data Science, AI Residency

In this notebook we build an AI agent that helps a security analyst investigate an incident.
It has **two tools**:

| Tool | What it does | Answers the question |
|---|---|---|
| `security_playbook` | Searches our incident-response playbook | *What are the rules?* |
| `query_security_logs` | Runs a read-only SQL query on our security logs | *What actually happened?* |

All data in this notebook is **synthetic** (made up for training).

## Step 0: Add your OpenAI key to Colab Secrets (one time only)

Never paste your API key directly into a notebook. Colab has a safe place for it called **Secrets**.

1. Click the **key icon** on the left sidebar of Colab.
2. Click **+ Add new secret**.
3. In **Name**, type exactly: `OPENAI_API_KEY`
4. In **Value**, paste your OpenAI key (it starts with `sk-`).
5. Switch **Notebook access** ON.

That's it. The code below reads the key from Secrets, so it never appears in your notebook.

## Step 1: Install the libraries (takes about 1 minute)

In [1]:
# langchain        -> the agent framework (create_agent lives here)
# langchain-openai -> lets LangChain talk to OpenAI models
# langgraph        -> the runtime create_agent uses underneath (also gives us memory)
!pip install -q -U "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 8.7 MB/s eta 0:00:00


## Step 2: Load the API key from Colab Secrets

In [2]:
import os
from google.colab import userdata

try:
    # Read the secret you saved in Step 0 and put it where LangChain expects it
    os.environ["OPENAI_API_KEY"] = userdata.get("openai")
    print("OpenAI key loaded from Colab Secrets.")
except userdata.SecretNotFoundError:
    print("No secret called OPENAI_API_KEY found. Go back to Step 0 and add it.")
except userdata.NotebookAccessError:
    print("The secret exists, but Notebook access is OFF. Switch it ON in the key panel.")

MODEL = "gpt-4.1-mini"   # a fast, low-cost model

OpenAI key loaded from Colab Secrets.


## Step 3: Create the security logs (our database)

We create a small SQLite database with one table, `security_events`.
It has normal background activity **plus one suspicious incident** hidden inside it.

In [19]:
import sqlite3
import random

random.seed(13)   # same random data every time we run

# --- 1. Normal background activity (the "noise" a SOC sees every day) ---
hosts = ["ENG-WS-03", "ENG-WS-07", "HR-LT-21", "FIN-LT-08", "OPS-WS-02"]
users = ["a.khan", "o.haddad", "r.sharma", "p.patel", "m.chen"]
normal_events = [
    ("auth",     "login_success",      "Low",    "Normal login"),
    ("auth",     "login_failed",       "Low",    "Wrong password"),
    ("firewall", "connection_blocked", "Medium", "Blocked inbound scan"),
    ("endpoint", "av_detection",       "Medium", "Adware quarantined"),
]

rows = []
for i in range(400):
    source, event_type, severity, details = random.choice(normal_events)
    hour, minute = random.randint(0, 23), random.randint(0, 59)
    timestamp = f"2026-10-09 {hour:02d}:{minute:02d}:00"
    rows.append((timestamp, source, random.choice(hosts), random.choice(users), "10.10.4.21", "IT", event_type, severity, details))

# --- 2. The incident on the night of 9 to 10 October ---
attacker_ip = "45.155.205.99"   # an external IP address
for n in range(1, 7):           # six failed logins in a row
    rows.append((f"2026-10-10 01:4{n}:00", "auth", "ENG-WS-07", "svc_historian", attacker_ip, "IT", "login_failed", "Low", f"Wrong password (attempt {n})"))

rows += [
    ("2026-10-10 01:52:05", "auth",       "ENG-WS-07", "svc_historian", attacker_ip,  "IT", "login_success",      "Medium", "Interactive login by service account"),
    ("2026-10-10 01:58:12", "firewall",   "JMP-02",    "svc_historian", "10.10.4.37", "OT", "connection_allowed", "Low",    "IT to OT session via jump host"),
    ("2026-10-10 02:03:44", "endpoint",   "ENG-WS-07", "svc_historian", "10.10.4.37", "IT", "usb_mounted",        "Medium", "USB storage device mounted"),
    ("2026-10-10 02:09:27", "ot_monitor", "PLC-CW-01", "svc_historian", "10.10.4.37", "OT", "plc_write",          "High",   "Write command to cooling-water pump controller"),
    ("2026-10-10 02:11:03", "historian",  "A-001",     "svc_historian", "10.10.4.37", "OT", "setpoint_change",    "High",   "Pump A-001 speed changed 1450 to 1720 rpm"),
]

# --- 3. Save everything into a SQLite database file called soc.db ---
con = sqlite3.connect("soc.db")
con.execute("DROP TABLE IF EXISTS security_events")
con.execute("""CREATE TABLE security_events (
    timestamp TEXT, source TEXT, host TEXT, user TEXT,
    src_ip TEXT, zone TEXT, event_type TEXT, severity TEXT, details TEXT)""")
con.executemany("INSERT INTO security_events VALUES (?,?,?,?,?,?,?,?,?)", rows)
con.commit()
con.close()
print(f"Created soc.db with {len(rows)} security events")

Created soc.db with 411 security events


In [21]:
import sqlite3
import pandas as pd

# Connect to the database and retrieve all security events
con = sqlite3.connect("soc.db")
df = pd.read_sql("SELECT * FROM security_events", con)
con.close()

# Display the entire dataset
pd.set_option('display.max_rows', None)
df.head(5)

,timestamp,source,host,user,src_ip,zone,event_type,severity,details
0,2026-10-09 09:43:00,firewall,ENG-WS-07,o.haddad,10.10.4.21,IT,connection_blocked,Medium,Blocked inbound scan
1,2026-10-09 07:41:00,auth,ENG-WS-07,o.haddad,10.10.4.21,IT,login_failed,Low,Wrong password
2,2026-10-09 17:54:00,auth,ENG-WS-07,r.sharma,10.10.4.21,IT,login_success,Low,Normal login
3,2026-10-09 13:08:00,auth,OPS-WS-02,a.khan,10.10.4.21,IT,login_success,Low,Normal login
4,2026-10-09 04:05:00,firewall,HR-LT-21,p.patel,10.10.4.21,IT,connection_blocked,Medium,Blocked inbound scan


In [23]:
# Peek at the last 8 events (this is where the incident is)
import pandas as pd
pd.read_sql("SELECT * FROM security_events ORDER BY timestamp DESC LIMIT 15", sqlite3.connect("soc.db"))

,timestamp,source,host,user,src_ip,zone,event_type,severity,details
0,2026-10-10 02:11:03,historian,A-001,svc_historian,10.10.4.37,OT,setpoint_change,High,Pump A-001 speed changed 1450 to 1720 rpm
1,2026-10-10 02:09:27,ot_monitor,PLC-CW-01,svc_historian,10.10.4.37,OT,plc_write,High,Write command to cooling-water pump controller
2,2026-10-10 02:03:44,endpoint,ENG-WS-07,svc_historian,10.10.4.37,IT,usb_mounted,Medium,USB storage device mounted
3,2026-10-10 01:58:12,firewall,JMP-02,svc_historian,10.10.4.37,OT,connection_allowed,Low,IT to OT session via jump host
4,2026-10-10 01:52:05,auth,ENG-WS-07,svc_historian,45.155.205.99,IT,login_success,Medium,Interactive login by service account
5,2026-10-10 01:46:00,auth,ENG-WS-07,svc_historian,45.155.205.99,IT,login_failed,Low,Wrong password (attempt 6)
6,2026-10-10 01:45:00,auth,ENG-WS-07,svc_historian,45.155.205.99,IT,login_failed,Low,Wrong password (attempt 5)
7,2026-10-10 01:44:00,auth,ENG-WS-07,svc_historian,45.155.205.99,IT,login_failed,Low,Wrong password (attempt 4)
8,2026-10-10 01:43:00,auth,ENG-WS-07,svc_historian,45.155.205.99,IT,login_failed,Low,Wrong password (attempt 3)
9,2026-10-10 01:42:00,auth,ENG-WS-07,svc_historian,45.155.205.99,IT,login_failed,Low,Wrong password (attempt 2)


## Step 4: The incident-response playbook (our document)

This is the text our first tool will search. In real life this would be a PDF or a wiki page.

In [24]:
PLAYBOOK = """
SOC-PB-001  Incident Response Playbook (Energy Plant)

1. Severity levels
Low: review in daily triage. Medium: investigate within 4 hours.
High: investigate within 30 minutes. Critical: respond immediately and page the OT on-call engineer.

2. Brute force
Five or more failed logins for the same account within 10 minutes is a suspected brute force attack.
If a successful login follows the failed attempts, treat it as a suspected credential compromise (High).
Logins from external IP addresses to internal workstations raise severity by one level.

3. Service accounts
Service accounts (names starting with svc_) must never log in interactively. Any interactive login by a service account is High.

4. IT to OT access
IT to OT access is only allowed through the jump host JMP-02, during maintenance windows:
Tuesday and Thursday, 10:00 to 14:00. Any IT to OT session outside a maintenance window is High.

5. USB devices
USB storage is not allowed on engineering workstations (ENG-WS-*). A USB mount on these hosts is Medium, or High if combined with other alerts.

6. OT changes
Any PLC write or setpoint change outside a maintenance window is Critical. Page the OT on-call engineer and verify with plant operations.

7. Containment
Isolating a host, disabling an account or blocking an IP needs approval from the SOC lead.
OT assets must never be isolated or shut down without approval from plant operations.
Never delete or change logs during an investigation.
"""
print("Playbook loaded:", len(PLAYBOOK), "characters")

Playbook loaded: 1467 characters


## Step 5: Why an LLM alone is not enough

Let's ask a normal LLM about our incident. It has never seen our logs or our playbook.

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)   # temperature=0 means "be consistent"
answer = llm.invoke("Was there suspicious activity on host ENG-WS-07 last night? What does our playbook say?")
print(answer.content)

I don’t have direct access to your security logs or systems to check for suspicious activity on host ENG-WS-07. However, I can help you understand what your playbook likely advises in such a situation.

Typically, a security playbook for investigating suspicious activity on a host like ENG-WS-07 would include steps such as:

1. **Initial Assessment:**
   - Review alerts and logs from the host for unusual activity (e.g., failed login attempts, unexpected processes, network connections).
   - Check endpoint detection and response (EDR) tools for any flagged behavior.

2. **Containment:**
   - Isolate the host from the network if active compromise is suspected.
   - Disable affected user accounts if credential compromise is suspected.

3. **Investigation:**
   - Collect forensic data (memory dump, disk images, logs).
   - Identify the scope and impact of the suspicious activity.
   - Determine the attack vector and timeline.

4. **Eradication:**
   - Remove malware or unauthorized access.

**Notice:** the answer is generic. The model cannot see our data. The fix is **tools**, not a better prompt.

## Step 6: Build Tool 1, `security_playbook`

A tool is just a **Python function** with a **description**.
The `@tool` decorator turns it into something the agent can use.
The description (the text in triple quotes) tells the model **when** to use the tool, so write it clearly.

In [10]:
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Cut the playbook into small chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents([Document(page_content=PLAYBOOK)])

# 2. Turn each chunk into numbers (embeddings) and store them so we can search by meaning
store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
print(f"Playbook split into {len(chunks)} searchable chunks")

# 3. The tool: find the 3 most relevant chunks for a question
@tool
def security_playbook(question: str) -> str:
    """Search the SOC incident-response playbook for rules: severity, brute force,
    service accounts, IT to OT access, maintenance windows, USB, OT changes, containment."""
    results = store.similarity_search(question, k=3)
    return "\n---\n".join(doc.page_content for doc in results)

Playbook split into 5 searchable chunks


In [25]:
# Test the tool on its own, before any agent
print(security_playbook.invoke({"question": "Can a service account log in interactively?"}))

3. Service accounts
Service accounts (names starting with svc_) must never log in interactively. Any interactive login by a service account is High.

4. IT to OT access
IT to OT access is only allowed through the jump host JMP-02, during maintenance windows:
Tuesday and Thursday, 10:00 to 14:00. Any IT to OT session outside a maintenance window is High.
---
2. Brute force
Five or more failed logins for the same account within 10 minutes is a suspected brute force attack.
If a successful login follows the failed attempts, treat it as a suspected credential compromise (High).
Logins from external IP addresses to internal workstations raise severity by one level.
---
7. Containment
Isolating a host, disabling an account or blocking an IP needs approval from the SOC lead.
OT assets must never be isolated or shut down without approval from plant operations.
Never delete or change logs during an investigation.


## Step 7: Build Tool 2, `query_security_logs`

This tool runs a SQL query on our logs. Because the **model writes the SQL**, we protect it:
- it only accepts a single `SELECT` query
- the database is opened in **read-only** mode, so nothing can ever be changed

In [26]:
@tool
def query_security_logs(sql: str) -> str:
    """Run ONE read-only SQL SELECT query on the security logs.
    Table: security_events(timestamp, source, host, user, src_ip, zone, event_type, severity, details).
    timestamp format: 'YYYY-MM-DD HH:MM:SS'. Use this for what actually happened."""

    # Guard 1: only allow a single SELECT query
    query = sql.strip().rstrip(";")
    blocked_words = ["insert", "update", "delete", "drop", "alter", "create"]
    if not query.lower().startswith("select") or ";" in query or any(w in query.lower() for w in blocked_words):
        return "REJECTED: only one read-only SELECT query is allowed."

    # Guard 2: open the database in read-only mode
    con = sqlite3.connect("file:soc.db?mode=ro", uri=True)
    try:
        rows = con.execute(query).fetchall()
        return str(rows[:30]) if rows else "No rows found."
    except Exception as error:
        return f"SQL error: {error}"
    finally:
        con.close()

In [27]:
# Test the tool on its own
print(query_security_logs.invoke({"sql": "SELECT COUNT(*) FROM security_events WHERE event_type = 'login_failed'"}))

[(105,)]


## Step 8: What the model actually sees

The model never runs our Python code. It only sees the tool's **name, description and inputs** as JSON.
When it wants to use a tool, it writes back a small JSON request.

In [28]:
llm_with_tools = llm.bind_tools([security_playbook, query_security_logs])
reply = llm_with_tools.invoke("How many failed logins did svc_historian have on ENG-WS-07?")

print("Text answer:", repr(reply.content))   # empty: the model did not answer yet
print("Tool request:", reply.tool_calls)     # instead, it asks us to run a tool

Text answer: ''
Tool request: [{'name': 'query_security_logs', 'args': {'sql': "SELECT COUNT(*) AS failed_logins FROM security_events WHERE user = 'svc_historian' AND host = 'ENG-WS-07' AND event_type = 'failed_login'"}, 'id': 'call_eiPmzZJj50NosbSzJIoplP30', 'type': 'tool_call'}]


**Key idea:** the model asked for a tool, but nobody ran it yet.
Running the tool, sending back the result, and letting the model decide the next step is the **agent loop**.
`create_agent` does all of that for us.

## Step 9: Build the agent with `create_agent`

We give it four things: a **model**, our **tools**, a **system prompt** (its job description), and a **checkpointer** (its memory).

In [29]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = """You are the SOC Assistant for an energy plant.
- Use security_playbook for rules. Never invent rules.
- Use query_security_logs to see what actually happened.
- For an investigation: build a timeline, check each event against the playbook,
  give an overall severity, and list next steps with who must approve them.
- Always show evidence (times, hosts, users).
- You are READ-ONLY. You cannot isolate hosts, disable accounts or change logs. You can only recommend."""

soc_agent = create_agent(
    model=ChatOpenAI(model=MODEL, temperature=0),
    tools=[security_playbook, query_security_logs],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),   # memory: remembers each conversation by its thread_id
)
print("SOC Assistant is ready")

SOC Assistant is ready


In [16]:
# A small helper so we can watch every step the agent takes
def ask(question, thread_id="demo"):
    config = {"configurable": {"thread_id": thread_id}}   # same thread_id = same conversation
    for step in soc_agent.stream({"messages": [{"role": "user", "content": question}]}, config, stream_mode="updates"):
        for update in step.values():
            for message in (update or {}).get("messages", []):
                message.pretty_print()   # shows the AI's tool calls, the tool results and the final answer

## Step 10: Live demos

Watch which tool the agent picks for each question.

### Demo 1: a rules question (should use the playbook)

In [30]:
ask("Is IT to OT access allowed at 2 AM on a Saturday? Who must approve isolating an OT asset?", thread_id="d1")

================================== Ai Message ==================================
Tool Calls:
  security_playbook (call_F2DHVyr2haaF2Z2HH289480R)
 Call ID: call_F2DHVyr2haaF2Z2HH289480R
  Args:
    question: Is IT to OT access allowed at 2 AM on a Saturday?
  security_playbook (call_B2JK62nmHiZbuYwE4DuIuK1h)
 Call ID: call_B2JK62nmHiZbuYwE4DuIuK1h
  Args:
    question: Who must approve isolating an OT asset?
================================= Tool Message =================================
Name: security_playbook

7. Containment
Isolating a host, disabling an account or blocking an IP needs approval from the SOC lead.
OT assets must never be isolated or shut down without approval from plant operations.
Never delete or change logs during an investigation.
---
5. USB devices
USB storage is not allowed on engineering workstations (ENG-WS-*). A USB mount on these hosts is Medium, or High if combined with other alerts.

6. OT changes
Any PLC write or setpoint change outside a maintenance windo

### Demo 2: a data question (should use the logs)

In [31]:
ask("Which hosts had the most failed logins? Show the top 5.", thread_id="d2")

================================== Ai Message ==================================
Tool Calls:
  query_security_logs (call_dTHpFVv7svdFOrXIB3B5vTPy)
 Call ID: call_dTHpFVv7svdFOrXIB3B5vTPy
  Args:
    sql: SELECT host, COUNT(*) AS failed_logins FROM security_events WHERE event_type = 'failed_login' GROUP BY host ORDER BY failed_logins DESC LIMIT 5;
================================= Tool Message =================================
Name: query_security_logs

No rows found.
================================== Ai Message ==================================

There are no records of failed login attempts in the security logs. If you want, I can check for other types of suspicious login activity or events.


### Demo 3: a full investigation (should use BOTH tools)

In [32]:
ask("Investigate host ENG-WS-07 between 01:30 and 02:30 on 2026-10-10. Build a timeline, check each event against the playbook, give a severity and next steps.", thread_id="d3")

================================== Ai Message ==================================
Tool Calls:
  query_security_logs (call_Ee1ABB0XfBE1b1YR661Y70KX)
 Call ID: call_Ee1ABB0XfBE1b1YR661Y70KX
  Args:
    sql: SELECT timestamp, source, host, user, src_ip, zone, event_type, severity, details FROM security_events WHERE host = 'ENG-WS-07' AND timestamp BETWEEN '2026-10-10 01:30:00' AND '2026-10-10 02:30:00' ORDER BY timestamp ASC;
================================= Tool Message =================================
Name: query_security_logs

[('2026-10-10 01:41:00', 'auth', 'ENG-WS-07', 'svc_historian', '45.155.205.99', 'IT', 'login_failed', 'Low', 'Wrong password (attempt 1)'), ('2026-10-10 01:42:00', 'auth', 'ENG-WS-07', 'svc_historian', '45.155.205.99', 'IT', 'login_failed', 'Low', 'Wrong password (attempt 2)'), ('2026-10-10 01:43:00', 'auth', 'ENG-WS-07', 'svc_historian', '45.155.205.99', 'IT', 'login_failed', 'Low', 'Wrong password (attempt 3)'), ('2026-10-10 01:44:00', 'auth', 'ENG-WS-07', 'sv

### Demo 4: connecting the dots across sources

In [33]:
ask("Did any PLC write or setpoint change happen outside a maintenance window? Which user and IP were involved?", thread_id="d4")

================================== Ai Message ==================================
Tool Calls:
  security_playbook (call_Pko5Vd3PNvwA0wRXtIgl4pcA)
 Call ID: call_Pko5Vd3PNvwA0wRXtIgl4pcA
  Args:
    question: PLC write or setpoint change outside maintenance window
================================= Tool Message =================================
Name: security_playbook

5. USB devices
USB storage is not allowed on engineering workstations (ENG-WS-*). A USB mount on these hosts is Medium, or High if combined with other alerts.

6. OT changes
Any PLC write or setpoint change outside a maintenance window is Critical. Page the OT on-call engineer and verify with plant operations.
---
3. Service accounts
Service accounts (names starting with svc_) must never log in interactively. Any interactive login by a service account is High.

4. IT to OT access
IT to OT access is only allowed through the jump host JMP-02, during maintenance windows:
Tuesday and Thursday, 10:00 to 14:00. Any IT to OT sessi

## Step 11: Memory

Same `thread_id` = the agent remembers the conversation. A new `thread_id` = a fresh start.

In [ ]:
ask("What happened on ENG-WS-07 after 01:45 on 2026-10-10?", thread_id="memory-demo")

In [ ]:
ask("Which account was involved, and is it allowed to log in interactively?", thread_id="memory-demo")

In [ ]:
ask("Which account was involved?", thread_id="brand-new")   # no memory: it will not know what we mean

## Step 12: Security, three layers of protection

| Layer | Where | What it does |
|---|---|---|
| 1. Prompt | System prompt | Tells the agent it is read-only (helpful, but not enough alone) |
| 2. Tool guard | Inside the function | Rejects anything that is not a single SELECT |
| 3. Infrastructure | Database connection | Opened read-only, so nothing can be written |

In [34]:
# Layer 2 in action: these are rejected before they ever reach the database
print(query_security_logs.invoke({"sql": "DELETE FROM security_events"}))
print(query_security_logs.invoke({"sql": "SELECT * FROM security_events; DROP TABLE security_events"}))

REJECTED: only one read-only SELECT query is allowed.
REJECTED: only one read-only SELECT query is allowed.


In [35]:
# Now ask the agent to do something it is not allowed to do
ask("Isolate ENG-WS-07, disable svc_historian, and delete the failed login records.", thread_id="security")

================================== Ai Message ==================================

I am unable to isolate hosts, disable accounts, or delete records as I have read-only access only. However, I can investigate the situation for you, provide a detailed analysis, and recommend appropriate actions based on the security playbook.

Would you like me to start an investigation on ENG-WS-07 and the svc_historian account, including any failed login attempts?


## Assignment

1. Add a third read-only tool, `get_asset_info(host)`, that returns the owner, zone (IT or OT) and criticality of a host from a new `assets` table.
2. Ask: *"Which of last night's alerts touched a critical OT asset, and who must be paged?"*
3. Try to break the SQL guard with a tricky query. Which layer stopped you?
4. **Stretch:** create an `isolate_host` tool and add human approval with
   `from langchain.agents.middleware import HumanInTheLoopMiddleware`.

#Step 13 : Making a UI Gradio

In [18]:
!pip install -q gradio

import gradio as gr

def respond(question, thread_id):
    if not question.strip():
        return "Please enter a valid question."

    config = {"configurable": {"thread_id": thread_id or "gradio-demo"}}
    response_log = []

    # Stream updates from the agent
    try:
        for step in soc_agent.stream({"messages": [{"role": "user", "content": question}]}, config, stream_mode="updates"):
            for update in step.values():
                for message in (update or {}).get("messages", []):
                    # Collect the agent's textual updates
                    if message.content:
                        response_log.append(message.content)
                    elif hasattr(message, 'tool_calls') and message.tool_calls:
                        for tool in message.tool_calls:
                            response_log.append(f"* [Agent calling tool '{tool['name']}' with args: {tool['args']}] *\n")
    except Exception as e:
        return f"An error occurred: {e}"

    return "\n\n".join(response_log) if response_log else "No response generated."

# Create a simple Gradio interface
demo = gr.Interface(
    fn=respond,
    inputs=[
        gr.Textbox(label="Enter your question (e.g., 'Investigate host ENG-WS-07 between 01:30 and 02:30 on 2026-10-10')", placeholder="Type here..."),
        gr.Textbox(label="Session / Thread ID", value="gradio-demo")
    ],
    outputs=gr.Textbox(label="SOC Assistant Response", interactive=False),
    title="🛡️ SOC Assistant UI",
    description="Ask the AI agent about playbooks, SQL log data, or run complete incident investigations.",
    flagging_mode="never"
)

# Launch the app
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4c05661c5170476302.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4c05661c5170476302.gradio.live


## Step 14: MC 14 `create_agent` or LangGraph?

| Situation | Use |
|---|---|
| One agent, a few tools, the model decides the steps | `create_agent` |
| Need memory, streaming, or approval on one tool | `create_agent` + middleware |
| Fixed steps that must always run in order | LangGraph |
| Branching, retries, multiple agents, multi-step approvals | LangGraph |

`create_agent` already runs on LangGraph. You only drop down when you need to control the loop yourself.